# 1) Dependency Check
This cell verifies that the required training packages are importable before running the fine-tuning workflow.

In [ ]:
import importlib

required_packages = ["peft", "datasets", "transformers", "accelerate"]
for package in required_packages:
    importlib.import_module(package)

print("✅ All required training dependencies are importable:", ", ".join(required_packages))

# 2) Load LIAR Dataset
This cell loads the LIAR dataset from Hugging Face using `load_dataset("liar")`.

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("liar")
raw_ds

# 3) Label Distribution + 6→3 Class Collapse
This cell prints the original LIAR label distribution and maps 6 labels into 3 classes:
- TRUE: `true`, `mostly-true` → 0
- HALF-TRUE: `half-true` → 1
- FALSE: `false`, `barely-true`, `pants-fire` → 2

In [ ]:
from collections import Counter

label_names = raw_ds["train"].features["label"].names

train_counts_6 = Counter(label_names[idx] for idx in raw_ds["train"]["label"])
print("Original 6-class train distribution:")
for name in label_names:
    print(f"  {name:12s}: {train_counts_6.get(name, 0)}")

collapse_map = {
    "true": 0,
    "mostly-true": 0,
    "half-true": 1,
    "false": 2,
    "barely-true": 2,
    "pants-fire": 2,
}

collapsed_names = {0: "TRUE", 1: "HALF-TRUE", 2: "FALSE"}

def collapse_example(example):
    label_name = label_names[example["label"]]
    example["label_3"] = collapse_map[label_name]
    return example

collapsed_ds = raw_ds.map(collapse_example)

train_counts_3 = Counter(collapsed_ds["train"]["label_3"])
print("\nCollapsed 3-class train distribution:")
for idx in [0, 1, 2]:
    print(f"  {collapsed_names[idx]:10s}: {train_counts_3.get(idx, 0)}")

# 4) Tokenization
This cell tokenizes LIAR statements with `DistilBertTokenizerFast` using `max_length=512` and truncation.

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
text_column = "statement"

def tokenize_batch(batch):
    return tokenizer(batch[text_column], truncation=True, max_length=512)

tokenized_ds = collapsed_ds.map(tokenize_batch, batched=True)
tokenized_ds = tokenized_ds.rename_column("label_3", "labels")
tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 5) Apply LoRA with PEFT
This cell initializes DistilBERT for 3-class sequence classification and applies LoRA using:
- `r=16`
- `lora_alpha=32`
- `target_modules=["q_lin", "v_lin"]`
- `task_type=SEQ_CLS`

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label={0: "TRUE", 1: "HALF-TRUE", 2: "FALSE"},
    label2id={"TRUE": 0, "HALF-TRUE": 1, "FALSE": 2},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 6) Train with Hugging Face Trainer
This cell trains for 3 epochs with batch size 16, evaluates each epoch, and computes macro F1 (`f1_macro`).

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir="models/checkpoints_liar",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
train_result

# 7) Print F1 Per Epoch
This cell extracts and prints validation macro F1 from Trainer logs for each epoch.

In [ ]:
epoch_f1 = []
for log_item in trainer.state.log_history:
    if "eval_f1_macro" in log_item and "epoch" in log_item:
        epoch_f1.append((log_item["epoch"], log_item["eval_f1_macro"]))

if not epoch_f1:
    print("No eval F1 logs found yet. Ensure training and evaluation completed.")
else:
    print("Validation F1 (macro) per epoch:")
    for epoch, f1_value in epoch_f1:
        print(f"  epoch {epoch}: {f1_value:.4f}")

# 8) Save LoRA Adapter
This cell saves the trained LoRA adapter to `models/liar_lora/`.

In [ ]:
import os

save_dir = "models/liar_lora"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Saved LoRA adapter (and tokenizer) to: {save_dir}")